In [ ]:
# Google Colab : télécharger les données de cette séance / download this session's data
import os, urllib.request
BASE = "https://raw.githubusercontent.com/Marzoug-Nabil/Nabil/main/academic/teaching/fsci6222/materials/s09/"
for f in ['heart.csv']:
    os.makedirs('data', exist_ok=True)
    for dst in (f, os.path.join('data', f)):
        if not os.path.exists(dst):
            urllib.request.urlretrieve(BASE + f, dst)


# FSCI 6222 : Analyse de donnees
# Seance 10 : Introduction au Deep Learning
## Notebook de demonstration : MLP et CNN avec Keras

**Universite de Moncton : Faculte des sciences : Ete 2026**

---

Ce notebook met en pratique les concepts de la seance 10 sur trois datasets :

1. **Heart Failure Prediction** (continuite S9) : MLP sur donnees tabulaires
2. **MNIST** : chiffres manuscrits, premier CNN
3. **Fashion-MNIST** : classification de vetements, CNN ameliore

### Pre-requis
- `pip install tensorflow scikit-learn matplotlib seaborn`
- Le fichier `heart.csv` doit etre dans le meme dossier (recuperable depuis l'exercice S9)


---

## 1. Installation et imports


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.datasets import mnist, fashion_mnist

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

sns.set_style('whitegrid')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

print(f'TensorFlow version : {tf.__version__}')
print(f'GPU disponible : {len(tf.config.list_physical_devices("GPU"))}')

---

## Partie 1 : MLP sur Heart Failure

On reprend le dataset de l'exercice integrateur de la S9, en appliquant la **meme preparation** mais en remplacant la regression logistique par un MLP.


### 2.1 Chargement et preparation des donnees


In [ ]:
df = pd.read_csv('heart.csv')

# Gerer les 0 invalides (NaN caches) - vu en S9
for col in ['Cholesterol', 'RestingBP']:
    median_val = df[df[col] > 0][col].median()
    df[col] = df[col].replace(0, median_val)

# Encodage des variables categorielles
df['Sex'] = df['Sex'].map({'M': 1, 'F': 0})
df['ExerciseAngina'] = df['ExerciseAngina'].map({'Y': 1, 'N': 0})
df = pd.get_dummies(df, columns=['ChestPainType', 'RestingECG', 'ST_Slope'],
                     drop_first=False, dtype=int)

X = df.drop(columns=['HeartDisease'])
y = df['HeartDisease']

print(f'Features : {X.shape[1]}, Patients : {X.shape[0]}')
print(f'Distribution cible : {y.value_counts().to_dict()}')

In [ ]:
# Split train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

# Standardisation (CRITIQUE pour les reseaux denses)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

print(f'Train : {X_train_s.shape}, Test : {X_test_s.shape}')

### 2.2 Construction du MLP


In [ ]:
# MLP avec 2 couches cachees + Dropout pour regularisation
model_mlp = keras.Sequential([
    layers.Input(shape=(X_train_s.shape[1],)),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(1, activation='sigmoid')  # classification binaire
])

model_mlp.summary()

In [ ]:
# Compilation
model_mlp.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# EarlyStopping pour eviter le surapprentissage
early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=15,
    restore_best_weights=True,
    verbose=1
)

### 2.3 Entrainement


In [ ]:
history_mlp = model_mlp.fit(
    X_train_s, y_train,
    epochs=200,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=0
)

print(f'\nEntrainement termine apres {len(history_mlp.history["loss"])} epochs')

In [ ]:
# Visualisation des courbes d'apprentissage
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(history_mlp.history['loss'], label='Train', color='steelblue')
axes[0].plot(history_mlp.history['val_loss'], label='Validation', color='crimson')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('Evolution de la loss')
axes[0].legend()

axes[1].plot(history_mlp.history['accuracy'], label='Train', color='steelblue')
axes[1].plot(history_mlp.history['val_accuracy'], label='Validation', color='crimson')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].set_title('Evolution de l\'accuracy')
axes[1].legend()

plt.tight_layout()
plt.show()

### 2.4 Evaluation et comparaison avec la regression logistique


In [ ]:
# Evaluation
test_loss, test_acc_mlp = model_mlp.evaluate(X_test_s, y_test, verbose=0)
y_pred_mlp = (model_mlp.predict(X_test_s, verbose=0) > 0.5).astype(int).flatten()

print(f'MLP : Test accuracy : {test_acc_mlp:.4f}')
print(f'\nMatrice de confusion :')
print(confusion_matrix(y_test, y_pred_mlp))

print(f'\nClassification report :')
print(classification_report(y_test, y_pred_mlp, target_names=['Sain', 'Malade']))

In [ ]:
# Comparaison directe avec regression logistique (S9)
from sklearn.linear_model import LogisticRegression

logreg = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
logreg.fit(X_train_s, y_train)
acc_logreg = logreg.score(X_test_s, y_test)

print('Comparaison MLP vs Regression logistique :')
print(f'  Regression logistique (S9) : {acc_logreg:.4f}')
print(f'  MLP (S10)                  : {test_acc_mlp:.4f}')
print(f'\nLECON CLE :')
print('Sur ce dataset tabulaire de petite taille (~900 patients),')
print('le MLP n\'apporte pas d\'amelioration significative.')
print('Le DL brille sur les donnees non structurees (images, audio, texte).')

---

## Partie 2 : CNN sur MNIST (chiffres manuscrits)

Le "hello world" du Deep Learning : reconnaitre des chiffres manuscrits.


> **Note d'execution** : les datasets MNIST et Fashion-MNIST sont telecharges automatiquement par Keras depuis Google Cloud Storage lors du premier appel a `mnist.load_data()`. Cela necessite une connexion Internet.
>
> Si vous executez ce notebook dans un environnement sans acces a Internet (ex : sandbox isolee), ces cellules echoueront. Pour les executer :
>
> - **Sur votre machine** : assurez-vous d'avoir une connexion Internet active.
> - **Sur Google Colab** : le telechargement marche automatiquement (GPU gratuit en bonus).
>
> La partie Heart Failure ci-dessus fonctionne hors-ligne car le CSV est local.


### 3.1 Chargement et exploration


In [ ]:
(X_train, y_train), (X_test, y_test) = mnist.load_data()

print(f'Train : {X_train.shape}, labels : {y_train.shape}')
print(f'Test  : {X_test.shape}, labels : {y_test.shape}')
print(f'Classes : {np.unique(y_train)}')
print(f'Pixels : min={X_train.min()}, max={X_train.max()}')

In [ ]:
# Visualiser quelques exemples
fig, axes = plt.subplots(2, 5, figsize=(11, 4.5))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_train[i], cmap='gray')
    ax.set_title(f'Label : {y_train[i]}')
    ax.axis('off')
plt.tight_layout()
plt.show()

### 3.2 Preparation des donnees pour CNN


In [ ]:
# Normaliser les pixels [0, 255] -> [0, 1]
X_train_n = X_train.astype('float32') / 255.0
X_test_n = X_test.astype('float32') / 255.0

# Ajouter la dimension canal (gris = 1 canal)
X_train_n = X_train_n.reshape(-1, 28, 28, 1)
X_test_n = X_test_n.reshape(-1, 28, 28, 1)

# One-hot encoding des labels
y_train_cat = to_categorical(y_train, 10)
y_test_cat = to_categorical(y_test, 10)

print(f'X_train shape : {X_train_n.shape}')
print(f'y_train_cat shape : {y_train_cat.shape}')
print(f'\nExemple one-hot : label {y_train[0]} -> {y_train_cat[0]}')

### 3.3 Construction et entrainement du CNN


In [ ]:
model_cnn = keras.Sequential([
    layers.Input(shape=(28, 28, 1)),
    
    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(10, activation='softmax')
])

model_cnn.summary()

In [ ]:
model_cnn.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

early_stop_cnn = keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=3, restore_best_weights=True
)

# Entrainement (5 epochs suffisent largement sur MNIST)
history_cnn = model_cnn.fit(
    X_train_n, y_train_cat,
    epochs=5,
    batch_size=128,
    validation_split=0.1,
    callbacks=[early_stop_cnn],
    verbose=1
)

### 3.4 Evaluation


In [ ]:
test_loss_cnn, test_acc_cnn = model_cnn.evaluate(X_test_n, y_test_cat, verbose=0)
print(f'CNN : Test accuracy MNIST : {test_acc_cnn:.4f}')

y_pred_cnn = model_cnn.predict(X_test_n, verbose=0).argmax(axis=1)

# Matrice de confusion
cm = confusion_matrix(y_test, y_pred_cnn)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=range(10), yticklabels=range(10), cbar=False)
plt.xlabel('Predit')
plt.ylabel('Reel')
plt.title(f'Matrice de confusion : MNIST (accuracy = {test_acc_cnn:.4f})')
plt.tight_layout()
plt.show()

In [ ]:
# Visualiser quelques erreurs
erreurs = np.where(y_pred_cnn != y_test)[0]
print(f'Nombre d\'erreurs : {len(erreurs)} sur 10000')
print(f'Accuracy : {(1 - len(erreurs)/10000):.4f}')

if len(erreurs) > 0:
    fig, axes = plt.subplots(2, 5, figsize=(11, 4.5))
    for i, ax in enumerate(axes.flat):
        if i < len(erreurs):
            idx = erreurs[i]
            ax.imshow(X_test[idx], cmap='gray')
            ax.set_title(f'Reel: {y_test[idx]}, Predit: {y_pred_cnn[idx]}',
                         color='red', fontsize=9)
            ax.axis('off')
    plt.suptitle('Exemples d\'erreurs du CNN')
    plt.tight_layout()
    plt.show()

---

## Partie 3 : CNN ameliore sur Fashion-MNIST

Meme format que MNIST, mais classification de vetements (plus difficile).


### 4.1 Chargement et exploration


In [ ]:
(X_train_fm, y_train_fm), (X_test_fm, y_test_fm) = fashion_mnist.load_data()

class_names = ['T-shirt', 'Pantalon', 'Pull', 'Robe', 'Manteau',
               'Sandale', 'Chemise', 'Basket', 'Sac', 'Bottine']

print(f'Train : {X_train_fm.shape}, Test : {X_test_fm.shape}')
print(f'Classes : {class_names}')

# Visualiser
fig, axes = plt.subplots(2, 5, figsize=(11, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_train_fm[i], cmap='gray')
    ax.set_title(class_names[y_train_fm[i]])
    ax.axis('off')
plt.tight_layout()
plt.show()

### 4.2 Preparation


In [ ]:
# Meme preprocessing que MNIST
X_train_fm_n = X_train_fm.astype('float32') / 255.0
X_test_fm_n = X_test_fm.astype('float32') / 255.0
X_train_fm_n = X_train_fm_n.reshape(-1, 28, 28, 1)
X_test_fm_n = X_test_fm_n.reshape(-1, 28, 28, 1)

y_train_fm_cat = to_categorical(y_train_fm, 10)
y_test_fm_cat = to_categorical(y_test_fm, 10)

### 4.3 CNN simple (baseline)


In [ ]:
# Meme architecture que pour MNIST
model_fm_simple = keras.Sequential([
    layers.Input(shape=(28, 28, 1)),
    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(10, activation='softmax')
])

model_fm_simple.compile(optimizer='adam',
                        loss='categorical_crossentropy',
                        metrics=['accuracy'])

early_stop_fm = keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=3, restore_best_weights=True
)

history_fm = model_fm_simple.fit(
    X_train_fm_n, y_train_fm_cat,
    epochs=10, batch_size=128,
    validation_split=0.1,
    callbacks=[early_stop_fm], verbose=1
)

_, acc_fm_simple = model_fm_simple.evaluate(X_test_fm_n, y_test_fm_cat, verbose=0)
print(f'\nCNN simple : Test accuracy Fashion-MNIST : {acc_fm_simple:.4f}')

### 4.4 CNN ameliore avec BatchNorm et data augmentation


In [ ]:
model_fm_v2 = keras.Sequential([
    layers.Input(shape=(28, 28, 1)),
    
    # Data augmentation legere
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.05),
    
    # Bloc 1
    layers.Conv2D(32, (3, 3), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),
    
    # Bloc 2
    layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),
    
    # Classification
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(10, activation='softmax')
])

model_fm_v2.compile(optimizer='adam',
                    loss='categorical_crossentropy',
                    metrics=['accuracy'])

history_fm_v2 = model_fm_v2.fit(
    X_train_fm_n, y_train_fm_cat,
    epochs=15, batch_size=128,
    validation_split=0.1,
    callbacks=[keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=5, restore_best_weights=True
    )], verbose=1
)

_, acc_fm_v2 = model_fm_v2.evaluate(X_test_fm_n, y_test_fm_cat, verbose=0)
print(f'\nCNN ameliore : Test accuracy : {acc_fm_v2:.4f}')
print(f'Gain par rapport au CNN simple : +{(acc_fm_v2 - acc_fm_simple)*100:.2f} points')

### 4.5 Analyse des erreurs : matrice de confusion


In [ ]:
y_pred_fm = model_fm_v2.predict(X_test_fm_n, verbose=0).argmax(axis=1)

cm_fm = confusion_matrix(y_test_fm, y_pred_fm)
plt.figure(figsize=(10, 8))
sns.heatmap(cm_fm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names, cbar=False)
plt.xlabel('Predit')
plt.ylabel('Reel')
plt.title(f'Matrice de confusion : Fashion-MNIST (accuracy = {acc_fm_v2:.4f})')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print('\nClasses les plus confondues :')
for i in range(10):
    for j in range(10):
        if i != j and cm_fm[i, j] > 30:
            print(f'  {class_names[i]:<10} confondu avec {class_names[j]:<10} : {cm_fm[i, j]} cas')

---

## 5. Synthese finale


In [ ]:
synthese = pd.DataFrame({
    'Dataset': ['Heart Failure', 'MNIST', 'Fashion-MNIST (simple)', 'Fashion-MNIST (ameliore)'],
    'Modele': ['MLP (3 couches)', 'CNN (2 blocs)', 'CNN (2 blocs)', 'CNN + BN + DataAug'],
    'Test accuracy': [f'{test_acc_mlp:.4f}', f'{test_acc_cnn:.4f}',
                      f'{acc_fm_simple:.4f}', f'{acc_fm_v2:.4f}'],
    'Comparaison ML': ['LogReg : 0.8913 (S9)', 'MLP : ~0.98', 'Difficile pour MLP', 'Difficile pour MLP'],
})
print(synthese.to_string(index=False))

## 6. Conclusions cles

### Ce qu'on a vu en pratique :

1. **MLP sur Heart Failure** : sans gain significatif vs regression logistique.
   - Confirme la regle : **DL brille sur donnees non structurees**
   - Sur tabulaire petite taille, ML classique reste excellent

2. **CNN sur MNIST** : ~99% d'accuracy avec une architecture simple.
   - Le "Hello World" du DL livre ses promesses
   - 2 blocs Conv+Pool suffisent

3. **CNN sur Fashion-MNIST** : montrer l'effet des techniques avancees.
   - CNN simple : ~91%
   - CNN + BatchNorm + Data Augmentation : ~93%
   - Classes les plus confondues : Pull/Manteau/Chemise (similitude visuelle)

### Lecons generales :

- **Standardisation obligatoire** pour les reseaux denses, normalisation [0,1] pour les images
- **EarlyStopping** : evite le surapprentissage et fait gagner du temps
- **Adam + ReLU** : choix par defaut moderne qui marche presque toujours
- **BatchNorm + Dropout + DataAug** : combinaison gagnante pour stabiliser et regulariser
- **Choisir le bon outil** : DL n'est pas toujours la reponse !

### Pour aller plus loin (S11) :

- Transfer learning avec modeles pre-entraines (VGG, ResNet, MobileNet)
- Tuning d'hyperparametres avec Keras Tuner
- Deploiement de modeles avec TensorFlow Serving ou ONNX

---

*Fin du notebook S10 : Introduction au Deep Learning*
